# Belka · 02 · Prata
Dado tratado: tipos corretos, nulos e inválidos tratados.

| Tabela | Granularidade | Chave do MERGE |
|---|---|---|
| `prata_clima_hora` | 1 linha por município por hora | `municipio_id` + `data_hora` |
| `prata_populacao` | 1 linha por município por ano | `municipio_id` + `ano` |

In [0]:
-- DROP TABLE IF EXISTS workspace.belka.prata_clima_hora;
-- DROP TABLE IF EXISTS workspace.belka.prata_populacao;
-- DROP TABLE IF EXISTS workspace.belka.prata_qualidade;
-- DROP TABLE IF EXISTS workspace.belka.prata_clima_quarentena;
-- DROP TABLE IF EXISTS workspace.belka.prata_populacao_quarentena;

In [0]:
%sql
-- Horas do Open-Meteo vêm em hora local de São Paulo ("2026-10-25T00:00").
SET TIME ZONE 'America/Sao_Paulo';

CREATE TABLE IF NOT EXISTS workspace.belka.prata_clima_hora (
  municipio_id     INT       COMMENT 'Código IBGE do município (7 dígitos)',
  municipio_nome   STRING,
  uf               STRING,
  latitude         DOUBLE,
  longitude        DOUBLE,
  data_hora        TIMESTAMP COMMENT 'Hora local America/Sao_Paulo',
  data             DATE,
  hora             INT,
  temperatura_c    DOUBLE,
  precipitacao_mm  DOUBLE    COMMENT 'Chuva acumulada na hora',
  umidade_pct      INT,
  rajada_kmh       DOUBLE,
  dia_coleta       DATE,
  coletado_em      TIMESTAMP,
  fonte            STRING,
  arquivo_origem   STRING,
  atualizado_em    TIMESTAMP
);

CREATE TABLE IF NOT EXISTS workspace.belka.prata_populacao (
  municipio_id     INT,
  municipio_nome   STRING,
  uf               STRING,
  ano              INT,
  populacao        BIGINT    COMMENT 'População estimada (IBGE, agregado 6579, variável 9324)',
  variavel_id      STRING,
  dia_coleta       DATE,
  coletado_em      TIMESTAMP,
  fonte            STRING,
  arquivo_origem   STRING,
  atualizado_em    TIMESTAMP
);

CREATE TABLE IF NOT EXISTS workspace.belka.prata_qualidade (
  executado_em  TIMESTAMP,
  tabela        STRING,
  regra         STRING,
  descricao     STRING,
  avaliados     BIGINT,
  barrados      BIGINT
);

## Regras de qualidade
Cada regra é escrita **uma vez**, como função, e usada no tratamento e no teste do fim do notebook.

| Regra | Tabela | Barra a linha quando | Por quê |
|---|---|---|---|
| R1 | clima | falta município ou hora, ou a hora não pertence ao dia coletado | sem chave não há MERGE nem alerta para um lugar e uma hora |
| R2 | clima | falta temperatura ou chuva | são as medidas que formam os níveis de risco |
| R3 | clima | valor fisicamente impossível: temperatura fora de −10 a 50 °C, chuva fora de 0 a 200 mm/h, umidade fora de 0 a 100 %, rajada fora de 0 a 250 km/h | erro de sensor ou de parsing não pode virar alerta |
| R4 | população | código do município ou ano ausente ou inválido | sem chave não há MERGE |
| R5 | população | população não numérica ou ≤ 0 (o IBGE usa `...` ou `-` para dado ausente) | a consulta de impacto soma pessoas |

**Nulos tratados sem barrar:** umidade e rajada ausentes ficam como `NULL`, porque não são essenciais
para os níveis de risco de chuva e calor. Textos recebem `trim` e a UF vai para maiúsculas.
As linhas barradas vão para tabelas de quarentena, com o motivo.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY FUNCTION belka_r1_chave_invalida(municipio_id INT, data_hora TIMESTAMP, dia_coleta DATE)
RETURNS BOOLEAN
RETURN municipio_id IS NULL OR data_hora IS NULL OR dia_coleta IS NULL OR to_date(data_hora) <> dia_coleta;

CREATE OR REPLACE TEMPORARY FUNCTION belka_r2_medida_ausente(temperatura DOUBLE, chuva DOUBLE)
RETURNS BOOLEAN
RETURN temperatura IS NULL OR chuva IS NULL;

CREATE OR REPLACE TEMPORARY FUNCTION belka_r3_fora_da_faixa(temperatura DOUBLE, chuva DOUBLE, umidade INT, rajada DOUBLE)
RETURNS BOOLEAN
RETURN coalesce(temperatura NOT BETWEEN -10 AND 50, false)
    OR coalesce(chuva       NOT BETWEEN 0 AND 200,  false)
    OR coalesce(umidade     NOT BETWEEN 0 AND 100,  false)
    OR coalesce(rajada      NOT BETWEEN 0 AND 250,  false);

CREATE OR REPLACE TEMPORARY FUNCTION belka_r4_chave_invalida(municipio_id INT, ano INT)
RETURNS BOOLEAN
RETURN municipio_id IS NULL OR municipio_id NOT BETWEEN 1000000 AND 9999999 OR ano IS NULL;

CREATE OR REPLACE TEMPORARY FUNCTION belka_r5_populacao_invalida(populacao BIGINT)
RETURNS BOOLEAN
RETURN populacao IS NULL OR populacao <= 0;

## Clima: abrir as listas horárias
Na Bronze, cada linha guarda um dia inteiro em 5 listas paralelas. `arrays_zip` junta as listas
posição a posição e `explode` transforma cada posição em uma linha: 1 dia → 24 horas.
`OUTER` mantém o dia mesmo se as listas vierem vazias, para a R1 barrar e contar em vez de sumir calada.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW clima_horas AS
WITH base AS (
  SELECT
    CAST(municipio_id AS INT)          AS municipio_id,
    trim(municipio_nome)               AS municipio_nome,
    upper(trim(uf))                    AS uf,
    latitude,
    longitude,
    try_cast(dia_coleta AS DATE)       AS dia_coleta,
    try_cast(coletado_em AS TIMESTAMP) AS coletado_em,
    fonte,
    arquivo_origem,
    hourly.time                 AS h_time,
    hourly.temperature_2m       AS h_temp,
    hourly.precipitation        AS h_chuva,
    hourly.relative_humidity_2m AS h_umid,
    hourly.wind_gusts_10m       AS h_rajada
  FROM workspace.belka.bronze_clima
),
horas AS (
  SELECT
    municipio_id, municipio_nome, uf, latitude, longitude,
    try_to_timestamp(z.h_time, "yyyy-MM-dd'T'HH:mm") AS data_hora,
    CAST(z.h_temp   AS DOUBLE) AS temperatura_c,
    CAST(z.h_chuva  AS DOUBLE) AS precipitacao_mm,
    CAST(z.h_umid   AS INT)    AS umidade_pct,
    CAST(z.h_rajada AS DOUBLE) AS rajada_kmh,
    dia_coleta, coletado_em, fonte, arquivo_origem
  FROM base
  LATERAL VIEW OUTER explode(arrays_zip(h_time, h_temp, h_chuva, h_umid, h_rajada)) t AS z
)
SELECT
  *,
  to_date(data_hora) AS data,
  hour(data_hora)    AS hora,
  belka_r1_chave_invalida(municipio_id, data_hora, dia_coleta)                AS r1,
  belka_r2_medida_ausente(temperatura_c, precipitacao_mm)                     AS r2,
  belka_r3_fora_da_faixa(temperatura_c, precipitacao_mm, umidade_pct, rajada_kmh) AS r3
FROM horas;

## População: abrir a série do IBGE
A série chega como `struct<2026:string>`: o ano é nome de campo e o valor é texto.
`to_json` + `from_json` transformam a struct em mapa ano → valor, que funciona para qualquer ano.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW populacao_linhas AS
WITH series AS (
  SELECT
    try_cast(p.dia_coleta AS DATE)       AS dia_coleta,
    try_cast(p.coletado_em AS TIMESTAMP) AS coletado_em,
    p.fonte, p.arquivo_origem,
    p.id AS variavel_id,
    s
  FROM workspace.belka.bronze_populacao p
  LATERAL VIEW OUTER explode(p.resultados[0].series) t AS s
),
anos AS (
  SELECT
    dia_coleta, coletado_em, fonte, arquivo_origem, variavel_id,
    s.localidade.id   AS loc_id,
    s.localidade.nome AS loc_nome,
    a.ano_txt, a.valor_txt
  FROM series
  LATERAL VIEW OUTER explode(from_json(to_json(s.serie), 'map<string,string>')) a AS ano_txt, valor_txt
),
tipado AS (
  SELECT
    try_cast(loc_id AS INT)                                       AS municipio_id,
    nullif(trim(regexp_extract(loc_nome, '^(.*) - [A-Z]{2}$', 1)), '') AS municipio_nome,
    nullif(regexp_extract(loc_nome, ' - ([A-Z]{2})$', 1), '')     AS uf,
    try_cast(ano_txt AS INT)                                      AS ano,
    try_cast(valor_txt AS BIGINT)                                 AS populacao,
    valor_txt,
    variavel_id, dia_coleta, coletado_em, fonte, arquivo_origem
  FROM anos
)
SELECT
  *,
  belka_r4_chave_invalida(municipio_id, ano) AS r4,
  belka_r5_populacao_invalida(populacao)     AS r5
FROM tipado;

## Contagem das regras
Um único `INSERT`, para todas as regras dividirem o mesmo `executado_em`.
Cada regra é contada de forma independente: uma linha pode falhar em mais de uma.

In [0]:
%sql
INSERT INTO workspace.belka.prata_qualidade
SELECT current_timestamp(), 'prata_clima_hora', 'R1', 'chave ausente ou hora fora do dia coletado',
       COUNT(*), COUNT_IF(r1) FROM clima_horas
UNION ALL
SELECT current_timestamp(), 'prata_clima_hora', 'R2', 'temperatura ou chuva ausente',
       COUNT(*), COUNT_IF(r2) FROM clima_horas
UNION ALL
SELECT current_timestamp(), 'prata_clima_hora', 'R3', 'valor fora da faixa física',
       COUNT(*), COUNT_IF(r3) FROM clima_horas
UNION ALL
SELECT current_timestamp(), 'prata_populacao', 'R4', 'município ou ano ausente ou inválido',
       COUNT(*), COUNT_IF(r4) FROM populacao_linhas
UNION ALL
SELECT current_timestamp(), 'prata_populacao', 'R5', 'população não numérica ou <= 0',
       COUNT(*), COUNT_IF(r5) FROM populacao_linhas;

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
SELECT tabela, regra, descricao, avaliados, barrados
FROM workspace.belka.prata_qualidade
WHERE executado_em = (SELECT MAX(executado_em) FROM workspace.belka.prata_qualidade)
ORDER BY regra;

tabela,regra,descricao,avaliados,barrados
prata_clima_hora,R1,chave ausente ou hora fora do dia coletado,1560,0
prata_clima_hora,R2,temperatura ou chuva ausente,1560,0
prata_clima_hora,R3,valor fora da faixa física,1560,0
prata_populacao,R4,município ou ano ausente ou inválido,65,0
prata_populacao,R5,população não numérica ou <= 0,65,0


## Quarentena
As linhas barradas ficam guardadas, com o motivo, para análise. Não entram na Prata.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.belka.prata_clima_quarentena AS
SELECT *, concat_ws(', ', CASE WHEN r1 THEN 'R1' END, CASE WHEN r2 THEN 'R2' END, CASE WHEN r3 THEN 'R3' END) AS motivos,
       current_timestamp() AS barrado_em
FROM clima_horas
WHERE r1 OR r2 OR r3;

CREATE OR REPLACE TABLE workspace.belka.prata_populacao_quarentena AS
SELECT *, concat_ws(', ', CASE WHEN r4 THEN 'R4' END, CASE WHEN r5 THEN 'R5' END) AS motivos,
       current_timestamp() AS barrado_em
FROM populacao_linhas
WHERE r4 OR r5;

num_affected_rows,num_inserted_rows


## MERGE: clima
Entram só as linhas aprovadas. No MERGE, uma linha que já existe só é atualizada se vier de uma coleta mais nova.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW lote_prata_clima AS
SELECT
  municipio_id, municipio_nome, uf, latitude, longitude,
  data_hora, data, hora,
  temperatura_c, precipitacao_mm, umidade_pct, rajada_kmh,
  dia_coleta, coletado_em, fonte, arquivo_origem,
  current_timestamp() AS atualizado_em
FROM clima_horas
WHERE NOT (r1 OR r2 OR r3)
QUALIFY row_number() OVER (PARTITION BY municipio_id, data_hora
                           ORDER BY coletado_em DESC, arquivo_origem DESC) = 1;

In [0]:
%sql
MERGE INTO workspace.belka.prata_clima_hora AS alvo
USING lote_prata_clima AS novo
ON  alvo.municipio_id = novo.municipio_id
AND alvo.data_hora    = novo.data_hora
WHEN MATCHED AND novo.coletado_em > alvo.coletado_em THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


## MERGE: população
Cada arquivo diário repete a mesma estimativa anual. Fica uma linha por município e ano, da coleta mais recente.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW lote_prata_populacao AS
SELECT
  municipio_id, municipio_nome, uf, ano, populacao, variavel_id,
  dia_coleta, coletado_em, fonte, arquivo_origem,
  current_timestamp() AS atualizado_em
FROM populacao_linhas
WHERE NOT (r4 OR r5)
QUALIFY row_number() OVER (PARTITION BY municipio_id, ano
                           ORDER BY coletado_em DESC, arquivo_origem DESC) = 1;

In [0]:
%sql
MERGE INTO workspace.belka.prata_populacao AS alvo
USING lote_prata_populacao AS novo
ON  alvo.municipio_id = novo.municipio_id
AND alvo.ano          = novo.ano
WHEN MATCHED AND novo.coletado_em > alvo.coletado_em THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


## Conferência

In [0]:
%sql
SELECT 'prata_clima_hora' AS tabela, COUNT(*) AS linhas, COUNT(DISTINCT municipio_id) AS municipios,
       CAST(MIN(data_hora) AS STRING) AS de, CAST(MAX(data_hora) AS STRING) AS ate
FROM workspace.belka.prata_clima_hora
UNION ALL
SELECT 'prata_populacao', COUNT(*), COUNT(DISTINCT municipio_id),
       CAST(MIN(ano) AS STRING), CAST(MAX(ano) AS STRING)
FROM workspace.belka.prata_populacao
UNION ALL
SELECT 'prata_clima_quarentena', COUNT(*), COUNT(DISTINCT municipio_id), NULL, NULL
FROM workspace.belka.prata_clima_quarentena
UNION ALL
SELECT 'prata_populacao_quarentena', COUNT(*), COUNT(DISTINCT municipio_id), NULL, NULL
FROM workspace.belka.prata_populacao_quarentena;

tabela,linhas,municipios,de,ate
prata_clima_hora,1560,5,2026-10-25 00:00:00,2026-11-06 23:00:00
prata_populacao,5,5,2026,2026
prata_clima_quarentena,0,0,null,null
prata_populacao_quarentena,0,0,null,null


In [0]:
%sql
-- Chave única: deve voltar vazio
SELECT municipio_id, data_hora, COUNT(*) AS vezes
FROM workspace.belka.prata_clima_hora
GROUP BY municipio_id, data_hora
HAVING COUNT(*) > 1;

municipio_id,data_hora,vezes


In [0]:
%sql
-- Amostra: um dia de Franco da Rocha, hora a hora
SELECT municipio_nome, data_hora, hora, temperatura_c, precipitacao_mm, umidade_pct, rajada_kmh
FROM workspace.belka.prata_clima_hora
WHERE municipio_id = 3516408
  AND data = (SELECT MAX(data) FROM workspace.belka.prata_clima_hora)
ORDER BY data_hora;

municipio_nome,data_hora,hora,temperatura_c,precipitacao_mm,umidade_pct,rajada_kmh
Franco da Rocha,2026-11-06T00:00:00.000-03:00,0,18.8,0.0,94,13.0
Franco da Rocha,2026-11-06T01:00:00.000-03:00,1,19.4,0.0,91,10.8
Franco da Rocha,2026-11-06T02:00:00.000-03:00,2,19.6,0.0,88,13.0
Franco da Rocha,2026-11-06T03:00:00.000-03:00,3,19.6,0.0,90,10.8
Franco da Rocha,2026-11-06T04:00:00.000-03:00,4,19.4,0.2,93,7.6
Franco da Rocha,2026-11-06T05:00:00.000-03:00,5,19.0,0.2,96,6.8
Franco da Rocha,2026-11-06T06:00:00.000-03:00,6,19.3,0.1,96,5.8
Franco da Rocha,2026-11-06T07:00:00.000-03:00,7,20.1,0.0,91,9.7
Franco da Rocha,2026-11-06T08:00:00.000-03:00,8,22.0,0.0,82,10.4
Franco da Rocha,2026-11-06T09:00:00.000-03:00,9,24.4,0.0,71,17.6


In [0]:
%sql
SELECT municipio_id, municipio_nome, uf, ano, populacao, dia_coleta
FROM workspace.belka.prata_populacao
ORDER BY populacao DESC;

municipio_id,municipio_nome,uf,ano,populacao,dia_coleta
3550308,null,null,2026,11911337,2026-11-06
3518800,null,null,2026,1351284,2026-11-06
3509502,null,null,2026,1189761,2026-11-06
3516408,null,null,2026,150648,2026-11-06
3550704,null,null,2026,84514,2026-11-06


## Teste das regras
Com dados reais é normal nenhuma linha ser barrada. Este teste passa linhas inventadas
(só em memória, nada é gravado) pelas **mesmas funções** e mostra que cada regra pega o que deve.

In [0]:
%sql
SELECT caso,
       belka_r1_chave_invalida(municipio_id, data_hora, dia_coleta)                AS r1,
       belka_r2_medida_ausente(temperatura_c, precipitacao_mm)                     AS r2,
       belka_r3_fora_da_faixa(temperatura_c, precipitacao_mm, umidade_pct, rajada_kmh) AS r3
FROM VALUES
  ('linha válida',          3550308, TIMESTAMP'2026-11-01 10:00', DATE'2026-11-01', 24.5, 0.0, 70,   18.0),
  ('hora ausente',          3550308, NULL,                        DATE'2026-11-01', 24.5, 0.0, 70,   18.0),
  ('hora de outro dia',     3550308, TIMESTAMP'2026-11-02 01:00', DATE'2026-11-01', 24.5, 0.0, 70,   18.0),
  ('temperatura ausente',   3550308, TIMESTAMP'2026-11-01 11:00', DATE'2026-11-01', NULL, 0.0, 70,   18.0),
  ('chuva negativa',        3550308, TIMESTAMP'2026-11-01 12:00', DATE'2026-11-01', 24.5, -3.0, 70,  18.0),
  ('umidade 140 %',         3550308, TIMESTAMP'2026-11-01 13:00', DATE'2026-11-01', 24.5, 0.0, 140,  18.0),
  ('umidade ausente (ok)',  3550308, TIMESTAMP'2026-11-01 14:00', DATE'2026-11-01', 24.5, 0.0, NULL, NULL)
  AS t(caso, municipio_id, data_hora, dia_coleta, temperatura_c, precipitacao_mm, umidade_pct, rajada_kmh);

caso,r1,r2,r3
linha válida,false,false,false
hora ausente,true,false,false
hora de outro dia,true,false,false
temperatura ausente,false,true,false
chuva negativa,false,false,true
umidade 140 %,false,false,true
umidade ausente (ok),false,false,false


In [0]:
%sql
SELECT caso,
       belka_r4_chave_invalida(municipio_id, ano) AS r4,
       belka_r5_populacao_invalida(try_cast(valor AS BIGINT)) AS r5
FROM VALUES
  ('linha válida',       3550308, 2026, '11895578'),
  ('código curto',       355030,  2026, '11895578'),
  ('valor "..." do IBGE', 3550308, 2026, '...'),
  ('população zero',     3550308, 2026, '0')
  AS t(caso, municipio_id, ano, valor);

caso,r4,r5
linha válida,false,false
código curto,true,false
"valor ""..."" do IBGE",false,true
população zero,false,true


## Histórico do MERGE
Em `operationMetrics`: linhas inseridas e atualizadas em cada execução.

In [0]:
%sql
DESCRIBE HISTORY workspace.belka.prata_clima_hora;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-07T16:28:05.000-03:00,76186718158206,fernandakaory2@gmail.com,MERGE,"Map(predicate -> [""((municipio_id#37942 = municipio_id#37806) AND (data_hora#37947 = data_hora#37811))""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(coletado_em#37819 > coletado_em#37955)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(251790475778572),ca3c5def-bdb1-47dc-a2ca-7819c8970ce6,1107-185316-8tejxtga-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2336, materializeSourceTimeMs -> 521, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 772, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1560, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 986)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-07T16:14:18.000-03:00,76186718158206,fernandakaory2@gmail.com,MERGE,"Map(predicate -> [""((municipio_id#30226 = municipio_id#30090) AND (data_hora#30231 = data_hora#30095))""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(coletado_em#30103 > coletado_em#30239)"",""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(251790475778572),62a0674a-4841-4892-8d13-7dd63bf055bc,1107-185316-8tejxtga-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 13553, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2242, materializeSourceTimeMs -> 555, numTargetRowsInserted -> 1560, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 655, numTargetRowsUpdated -> 0, numOutputRows -> 1560, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1560, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 987)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-07T16:13:48.000-03:00,76186718158206,fernandakaory2@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""2f988bd8-715e-4a44-b675-72e796f04d23"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-b5f60478-ea82-41f0-8b67-9ee4ea660827"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-63615226-f560-4dbb-a7b7-636c99cdf3c1""}, statsOnLoad -> false, clusterByAuto -> true)",null,List(251790475778572),f9ec1ac6-1aba-41fd-8b48-c2d5cd420bef,1107-185316-8tejxtga-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
